In [ ]:
import os, gc, warnings, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

MY_DATA_DIR = '/kaggle/input/datasets/min2602/my-data'
INPUT_DIR   = MY_DATA_DIR + '/'
OUTPUT_DIR  = '/kaggle/working/'
TARGET       = '임신 성공 여부'
RANDOM_STATE = 42
N_SPLITS     = 5

import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    USE_GPU = r.returncode == 0
    print('[GPU] 사용 가능' if USE_GPU else '[CPU] 모드')
except:
    USE_GPU = False

train_raw = pd.read_csv(INPUT_DIR + 'train.csv')
test_raw  = pd.read_csv(INPUT_DIR + 'test.csv')
y = train_raw[TARGET].astype(int)
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print(f'train: {train_raw.shape}, test: {test_raw.shape}')
print(f'scale_pos_weight: {scale_pos_weight:.3f}')


In [ ]:
AGE_INFO = {
    '만18-34세': {'median': 26.0,  'risk': 'Normal'},
    '만35-37세': {'median': 36.0,  'risk': 'High_Early'},
    '만38-39세': {'median': 38.5,  'risk': 'High_Early'},
    '만40-42세': {'median': 41.0,  'risk': 'High_Extreme'},
    '만43-44세': {'median': 43.5,  'risk': 'High_Extreme'},
    '만45-50세': {'median': 47.5,  'risk': 'High_Reversal'},
}
AGE_MEDIAN_FILLNA = 38.5

COLS_TO_DROP = [
    '임신 성공 여부',
    'ID', '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일',
    '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 농도', '불임 원인 - 정자 형태',
    'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부',
]

ISNA_FLAG_COLS = ['배아 해동 경과일', '난자 혼합 경과일', '배아 이식 경과일']
IQR_CLIP_COLS  = ['저장된 배아 수']
KNOWN_REASONS  = ['현재 시술용', '배아 저장용', '난자 저장용', '기증용', '연구용']

FILL_CAT = '알 수 없음'
UNK_CAT  = '__UNK__'

print('상수 정의 완료')


In [ ]:
class FoldPreprocessor:
    def __init__(self):
        self.iqr_bounds  = {}
        self.ohe         = None
        self._ohe_fitted = False

    @staticmethod
    def _classify_treatment(x):
        t = str(x).upper().strip()
        if 'BLASTOCYST' in t: return 'Blastocyst_Transfer'
        if 'ICSI'       in t: return 'ICSI'
        if 'IVF'        in t: return 'IVF'
        if 'IUI'        in t: return 'IUI'
        return 'Unknown'

    def fit(self, df_train: pd.DataFrame):
        for col in IQR_CLIP_COLS:
            if col in df_train.columns:
                q1 = df_train[col].quantile(0.25)
                q3 = df_train[col].quantile(0.75)
                self.iqr_bounds[col] = q3 + 1.5 * (q3 - q1)
        if '특정 시술 유형' in df_train.columns:
            series = df_train['특정 시술 유형'].fillna('').apply(
                self._classify_treatment).to_frame(name='시술_분류_그룹')
            try:
                self.ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
            except TypeError:
                self.ohe = OneHotEncoder(sparse=False, handle_unknown='ignore', drop='first')
            self.ohe.fit(series)
            self._ohe_fitted = True
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(columns=COLS_TO_DROP, errors='ignore').copy()

        for col in ISNA_FLAG_COLS:
            if col in df.columns:
                df[f'{col}_isna'] = df[col].isna().astype(np.int8)
        if '배아 이식 경과일_isna' in df.columns:
            df['배아이식_없음'] = df['배아 이식 경과일_isna'].copy()

        if '시술 당시 나이' in df.columns:
            df['Age_Median']     = df['시술 당시 나이'].map(lambda x: AGE_INFO.get(x, {}).get('median', None))
            df['Age_Risk_Group'] = df['시술 당시 나이'].map(lambda x: AGE_INFO.get(x, {}).get('risk', 'Unknown'))
            df = df.drop(columns=['시술 당시 나이'])

        age_med = df['Age_Median'].fillna(AGE_MEDIAN_FILLNA)

        total    = df.get('총 생성 배아 수',  pd.Series(0, index=df.index))
        transfer = df.get('이식된 배아 수',   pd.Series(0, index=df.index))
        df['배아_잉여율']       = np.where(total > 0, (total - transfer) / total, 0).clip(0, 1)
        df['Age×배아수']        = age_med * total

        collected = df.get('수집된 신선 난자 수', pd.Series(0, index=df.index))
        mixed     = df.get('혼합된 난자 수',      pd.Series(0, index=df.index))
        df['수정률']            = np.where(collected > 0, mixed / collected, 0).clip(0, 1)

        stored = df.get('저장된 배아 수', pd.Series(0, index=df.index))
        df['저장배아_보유여부'] = (stored > 0).astype(int)

        def parse_count(col_name):
            s = df.get(col_name, pd.Series('0회', index=df.index))
            return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)
        df['총_시술횟수'] = parse_count('IVF 시술 횟수') + parse_count('DI 시술 횟수')

        df['기증정자_사용여부'] = df.get(
            '기증자 정자와 혼합된 난자 수',
            pd.Series(np.nan, index=df.index)).notna().astype(int)

        if '난자 출처' in df.columns:
            df['기증난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
        else:
            df['기증난자_여부'] = 0
        df['기증난자×나이'] = df['기증난자_여부'] * age_med
        df['자가난자×나이'] = (1 - df['기증난자_여부']) * age_med

        is_extreme = df['Age_Risk_Group'].isin(['High_Extreme', 'High_Reversal'])
        is_normal  = df['Age_Risk_Group'] == 'Normal'
        thaw_isna  = df.get('배아 해동 경과일_isna', pd.Series(1, index=df.index))
        df['초고위험_기증난자_조합'] = (is_extreme & (df['기증난자_여부'] == 1)).astype(np.int8)
        df['고령_동결배아_조합']     = (is_extreme & (thaw_isna == 0)).astype(np.int8)
        df['정상군_첫시술']          = (is_normal & (df['총_시술횟수'] == 0)).astype(np.int8)

        pgs = df.get('PGS 시술 여부',             pd.Series(np.nan, index=df.index))
        pgd = df.get('PGD 시술 여부',             pd.Series(np.nan, index=df.index))
        pic = df.get('착상 전 유전 검사 사용 여부', pd.Series(np.nan, index=df.index))
        df['유전검사_시행여부'] = (
            pgs.notna().astype(int) | pgd.notna().astype(int) | pic.notna().astype(int)
        ).astype(np.int8)

        if '특정 시술 유형' in df.columns:
            t_str = df['특정 시술 유형'].fillna('').astype(str).str.upper()
            df['is_IVF']       = t_str.str.contains('IVF').astype(np.int8)
            df['is_ICSI']      = t_str.str.contains('ICSI').astype(np.int8)
            df['is_BLASTOCYST']= t_str.str.contains('BLASTOCYST').astype(np.int8)
            thaw_isna2 = df.get('배아 해동 경과일_isna', pd.Series(1, index=df.index))
            df['is_FER']       = (t_str.str.contains('FER') | (thaw_isna2 == 0)).astype(np.int8)

            male_factor  = df.get('불임 원인 - 남성 요인', pd.Series(0, index=df.index)).astype(float)
            df['ICSI_남성요인_조합'] = (df['is_ICSI'] * male_factor).astype(np.int8)

            stored_emb = df.get('저장된 배아 수', pd.Series(0, index=df.index)).astype(float)
            df['FER_저장배아_조합'] = (df['is_FER'] * stored_emb).astype(np.float32)

            transferred = df.get('이식된 배아 수', pd.Series(0, index=df.index)).astype(float)
            df['포배기_단일이식_조합'] = (df['is_BLASTOCYST'] * (transferred == 1.0)).astype(np.int8)

            has_pgt = df.get('유전검사_시행여부', pd.Series(0, index=df.index))
            df['PGT_나이_조합'] = (has_pgt * age_med).astype(np.float32)

            donor_sperm = df.get('기증정자_사용여부', pd.Series(0, index=df.index))
            di_count    = df.get('DI 시술 횟수', pd.Series('0', index=df.index)).astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)
            df['기증정자_DI_조합'] = (donor_sperm * (di_count > 0)).astype(np.int8)

        if '배아 생성 주요 이유' in df.columns:
            split_series = df['배아 생성 주요 이유'].fillna('').apply(
                lambda x: [v.strip() for v in x.split(',') if v.strip()])
            mlb = MultiLabelBinarizer(classes=KNOWN_REASONS)
            encoded = pd.DataFrame(
                mlb.fit_transform(split_series),
                columns=[f'목적_{c}' for c in KNOWN_REASONS],
                index=df.index)
            df = pd.concat([df.drop(columns=['배아 생성 주요 이유']), encoded], axis=1)

        if self._ohe_fitted and '특정 시술 유형' in df.columns:
            series  = df['특정 시술 유형'].fillna('').apply(
                self._classify_treatment).to_frame(name='시술_분류_그룹')
            ohe_arr = self.ohe.transform(series)
            try:
                feature_names = self.ohe.get_feature_names_out(['시술_분류_그룹'])
            except AttributeError:
                feature_names = self.ohe.get_feature_names(['시술_분류_그룹'])
            ohe_cols = [f'시술_{c}' for c in feature_names]
            ohe_df   = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
            df = pd.concat([df.drop(columns=['특정 시술 유형']), ohe_df], axis=1)

        for col, upper in self.iqr_bounds.items():
            if col in df.columns:
                df[col] = df[col].clip(upper=upper)

        cat_cols = df.select_dtypes(include='object').columns.tolist()
        df[cat_cols] = df[cat_cols].fillna(FILL_CAT)

        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[num_cols] = df[num_cols].fillna(-999)

        return df

print('FoldPreprocessor 정의 완료')


In [ ]:
def _align_columns(X_tr, X_val, X_te, fill_value=-999.0):
    cols  = list(X_tr.columns)
    X_val = X_val.reindex(columns=cols, fill_value=fill_value)
    X_te  = X_te.reindex(columns=cols, fill_value=fill_value)
    return X_tr.copy(), X_val.copy(), X_te.copy()


def _is_cat(s):
    return s.dtype == object or pd.api.types.is_categorical_dtype(s)


def _prepare_lgbm(X_tr, X_val, X_te):
    X_tr, X_val, X_te = _align_columns(X_tr, X_val, X_te)
    cat_cols = [c for c in X_tr.columns if _is_cat(X_tr[c])]
    for col in cat_cols:
        tr_s = X_tr[col].where(X_tr[col].notna(), FILL_CAT).astype(str)
        cats = list(pd.Index(tr_s.unique()).astype(str))
        if UNK_CAT not in cats:
            cats.append(UNK_CAT)
        for X in (X_tr, X_val, X_te):
            s = X[col].where(X[col].notna(), FILL_CAT).astype(str)
            s = s.where(s.isin(cats), UNK_CAT)
            X[col] = pd.Categorical(s, categories=cats)
    return X_tr, X_val, X_te, cat_cols


def _prepare_catboost(X_tr, X_val, X_te):
    X_tr, X_val, X_te = _align_columns(X_tr, X_val, X_te)
    cat_cols = [c for c in X_tr.columns if _is_cat(X_tr[c])]
    for col in cat_cols:
        for X in (X_tr, X_val, X_te):
            X[col] = X[col].where(X[col].notna(), FILL_CAT).astype(str)
    for col in X_tr.columns:
        if col not in cat_cols:
            for X in (X_tr, X_val, X_te):
                X[col] = pd.to_numeric(X[col], errors='coerce').astype('float32')
    return X_tr, X_val, X_te, cat_cols


print('dtype 헬퍼 정의 완료')


In [ ]:
def verify_no_leakage(train_raw, test_raw):
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    tr_idx, val_idx = next(skf.split(train_raw, y))
    pp_fold = FoldPreprocessor().fit(train_raw.iloc[tr_idx])
    pp_full = FoldPreprocessor().fit(train_raw)
    X_val_fold = pp_fold.transform(train_raw.iloc[val_idx])
    X_val_full = pp_full.transform(train_raw.iloc[val_idx])
    X_test     = pp_fold.transform(test_raw)
    print('[누수 검증] IQR 상한값 비교 (fold vs full):')
    for col in IQR_CLIP_COLS:
        v_fold = pp_fold.iqr_bounds.get(col, 'N/A')
        v_full = pp_full.iqr_bounds.get(col, 'N/A')
        status = '⚠️ 차이있음' if v_fold != v_full else '✅ 정상'
        print(f'  {col}: fold={v_fold:.2f}, full={v_full:.2f} {status}')
    print(f'[누수 검증] 피처 수 — fold: {X_val_fold.shape[1]}, full: {X_val_full.shape[1]}, test: {X_test.shape[1]}')
    assert X_val_fold.shape[1] == X_test.shape[1], f'피처 수 불일치: train={X_val_fold.shape[1]}, test={X_test.shape[1]}'
    print('[누수 검증] ✅ PASS')

verify_no_leakage(train_raw, test_raw)


In [ ]:
import json
with open(INPUT_DIR + 'lgbm_best_params_day4_5.json', 'r') as f:
    LGBM_PARAMS_BASE = json.load(f)

if USE_GPU:
    LGBM_PARAMS_BASE['device_type'] = 'gpu'
    LGBM_PARAMS_BASE['max_bin']     = 63

print('LGBM 파라미터 로드 완료:')
for k, v in LGBM_PARAMS_BASE.items():
    if k not in ('objective','metric','random_state','n_jobs','verbose',
                 'device_type','max_bin'):
        print(f'  {k}: {v}')


In [ ]:
CATBOOST_PARAMS_BASE = {
    'iterations':            2000,
    'learning_rate':         0.05,
    'depth':                 6,
    'l2_leaf_reg':           3,
    'eval_metric':           'AUC',
    'early_stopping_rounds': 100,
    'verbose':               0,
}
if USE_GPU:
    CATBOOST_PARAMS_BASE['task_type']        = 'GPU'
    CATBOOST_PARAMS_BASE['devices']          = '0'
    CATBOOST_PARAMS_BASE['scale_pos_weight'] = scale_pos_weight
else:
    CATBOOST_PARAMS_BASE['class_weights'] = [1.0, scale_pos_weight]

print('CatBoost 기본 파라미터 사용 (v4 기준)')
print(f"  iterations: {CATBOOST_PARAMS_BASE['iterations']}")
print(f"  learning_rate: {CATBOOST_PARAMS_BASE['learning_rate']}")
print(f"  depth: {CATBOOST_PARAMS_BASE['depth']}")


In [ ]:
def run_lgbm_oof_from_folds(X_tr_folds, y_tr_folds,
                             X_val_folds, y_val_folds,
                             X_test_base, params, tag='lgbm'):
    """
    사전 준비된 fold 데이터로 LGBM OOF 학습.
    내부에서 _prepare_lgbm 적용 → 원본 fold 데이터 불변.
    """
    n_splits   = len(X_tr_folds)
    oof_preds  = np.zeros(len(y), dtype=np.float32)
    test_preds = np.zeros(len(X_test_base), dtype=np.float32)
    t0         = time.time()

    for fold, (X_tr, y_tr, X_val, y_val) in enumerate(
            zip(X_tr_folds, y_tr_folds, X_val_folds, y_val_folds)):

        X_tr_p, X_val_p, X_te_p, cat_feats = _prepare_lgbm(
            X_tr.copy(), X_val.copy(), X_test_base.copy())

        model = lgb.LGBMClassifier(**params)
        try:
            model.fit(
                X_tr_p, y_tr,
                eval_set=[(X_val_p, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False),
                           lgb.log_evaluation(500)],
                categorical_feature=cat_feats or 'auto',
            )
        except Exception as e:
            if USE_GPU:
                print(f'  [{tag}] Fold {fold+1}: GPU 실패 → CPU fallback: {str(e)[:80]}')
                cpu_p = {k: v for k, v in params.items()
                         if k not in ('device_type', 'max_bin')}
                model = lgb.LGBMClassifier(**cpu_p)
                model.fit(
                    X_tr_p, y_tr,
                    eval_set=[(X_val_p, y_val)],
                    callbacks=[lgb.early_stopping(100, verbose=False),
                               lgb.log_evaluation(500)],
                    categorical_feature=cat_feats or 'auto',
                )
            else:
                raise

        val_idx = y_val.index
        oof_preds[val_idx]  = model.predict_proba(X_val_p)[:, 1]
        test_preds         += model.predict_proba(X_te_p)[:, 1].astype(np.float32) / n_splits

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        print(f'  [{tag}] Fold {fold+1}/{n_splits} AUC: {fold_auc:.5f} | '
              f'best_iter={model.best_iteration_} | {time.time()-t0:.1f}s')
        del X_tr_p, X_val_p, X_te_p, model
        gc.collect()

    auc = roc_auc_score(y, oof_preds)
    print(f'[{tag}] OOF AUC: {auc:.5f}')
    return oof_preds, test_preds, auc


def run_catboost_oof_from_folds(X_tr_folds, y_tr_folds,
                                X_val_folds, y_val_folds,
                                X_test_base, params, tag='catboost'):
    """
    사전 준비된 fold 데이터로 CatBoost OOF 학습.
    내부에서 _prepare_catboost 적용.
    """
    n_splits   = len(X_tr_folds)
    oof_preds  = np.zeros(len(y), dtype=np.float32)
    test_preds = np.zeros(len(X_test_base), dtype=np.float32)
    t0         = time.time()

    for fold, (X_tr, y_tr, X_val, y_val) in enumerate(
            zip(X_tr_folds, y_tr_folds, X_val_folds, y_val_folds)):

        X_tr_cb, X_val_cb, X_te_cb, cat_feats = _prepare_catboost(
            X_tr.copy(), X_val.copy(), X_test_base.copy())

        model = CatBoostClassifier(**params)
        try:
            model.fit(
                X_tr_cb, y_tr.values,
                eval_set=(X_val_cb, y_val.values),
                cat_features=cat_feats,
                verbose=0,
            )
        except Exception as e:
            if USE_GPU:
                print(f'  [{tag}] Fold {fold+1}: GPU 실패 → CPU fallback: {str(e)[:80]}')
                cpu_p = {k: v for k, v in params.items()
                         if k not in ('task_type', 'devices')}
                model = CatBoostClassifier(**cpu_p)
                model.fit(
                    X_tr_cb, y_tr.values,
                    eval_set=(X_val_cb, y_val.values),
                    cat_features=cat_feats,
                    verbose=0,
                )
            else:
                raise

        val_idx = y_val.index
        oof_preds[val_idx]  = model.predict_proba(X_val_cb)[:, 1]
        test_preds         += model.predict_proba(X_te_cb)[:, 1].astype(np.float32) / n_splits

        fold_auc  = roc_auc_score(y_val, oof_preds[val_idx])
        best_iter = model.get_best_iteration() if hasattr(model, 'get_best_iteration') else None
        print(f'  [{tag}] Fold {fold+1}/{n_splits} AUC: {fold_auc:.5f} | '
              f'best_iter={best_iter} | {time.time()-t0:.1f}s')
        del X_tr_cb, X_val_cb, X_te_cb, model
        gc.collect()

    auc = roc_auc_score(y, oof_preds)
    print(f'[{tag}] OOF AUC: {auc:.5f}')
    return oof_preds, test_preds, auc


print('OOF 함수 정의 완료')


In [ ]:
skf_init = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
_init_tr_idx, _ = next(iter(skf_init.split(train_raw, y)))
pp_init = FoldPreprocessor().fit(train_raw.iloc[_init_tr_idx])
X_test_base = pp_init.transform(test_raw)
print(f'X_test_base shape: {X_test_base.shape}')


In [ ]:
SEEDS = [42, 0, 1, 7, 2026]

def run_multiseed_ensemble(seeds, lgbm_params_base, cat_params_base):
    """
    여러 random seed로 LGBM + CatBoost를 각각 학습하고
    전체 예측값의 평균을 반환한다.
    """
    all_lgbm_oof  = np.zeros(len(y))
    all_lgbm_test = np.zeros(len(X_test_base))
    all_cat_oof   = np.zeros(len(y))
    all_cat_test  = np.zeros(len(X_test_base))

    for seed in seeds:
        print(f'\n{"="*50}')
        print(f'  Seed {seed} 학습 시작')
        print(f'{"="*50}')

        # seed별 파라미터 업데이트
        lgbm_p = {**lgbm_params_base, 'random_state': seed}
        cat_p  = {**cat_params_base, 'random_seed': seed}

        # seed별 fold 분할
        skf = StratifiedKFold(
            n_splits=N_SPLITS, shuffle=True, random_state=seed)
        skf_splits = list(skf.split(train_raw, y))

        # fold별 전처리 (seed별로 새로 생성, 각 fold는 train fold에서만 fit)
        X_tr_s, y_tr_s   = [], []
        X_val_s, y_val_s = [], []

        for fold, (tr_idx, val_idx) in enumerate(skf_splits):
            pp = FoldPreprocessor()
            pp.fit(train_raw.iloc[tr_idx])
            X_tr_s.append(pp.transform(train_raw.iloc[tr_idx]))
            y_tr_s.append(y.iloc[tr_idx])
            X_val_s.append(pp.transform(train_raw.iloc[val_idx]))
            y_val_s.append(y.iloc[val_idx])

        # test 전처리: 이 seed의 fold0 train 데이터로 fit
        pp_for_test = FoldPreprocessor()
        pp_for_test.fit(train_raw.iloc[skf_splits[0][0]])
        X_test_s = pp_for_test.transform(test_raw)
        X_test_s = X_test_s.reindex(
            columns=X_tr_s[0].columns, fill_value=-999)

        # LGBM 학습
        lgbm_oof_s, lgbm_test_s, lgbm_auc_s = run_lgbm_oof_from_folds(
            X_tr_s, y_tr_s, X_val_s, y_val_s,
            X_test_s, lgbm_p, tag=f'lgbm_seed{seed}')

        # CatBoost 학습
        cat_oof_s, cat_test_s, cat_auc_s = run_catboost_oof_from_folds(
            X_tr_s, y_tr_s, X_val_s, y_val_s,
            X_test_s, cat_p, tag=f'cat_seed{seed}')

        print(f'  Seed {seed} — LGBM: {lgbm_auc_s:.5f} | '
              f'CatBoost: {cat_auc_s:.5f}')

        # 누적
        all_lgbm_oof  += lgbm_oof_s  / len(seeds)
        all_lgbm_test += lgbm_test_s / len(seeds)
        all_cat_oof   += cat_oof_s   / len(seeds)
        all_cat_test  += cat_test_s  / len(seeds)

        # 메모리 정리
        del X_tr_s, X_val_s, X_test_s, pp_for_test
        gc.collect()

    return all_lgbm_oof, all_lgbm_test, all_cat_oof, all_cat_test


print('run_multiseed_ensemble 정의 완료')


In [ ]:
def find_best_weights(oof_list, test_list, y, labels, step=0.01):
    best_auc, best_w, best_test = 0, (0.5, 0.5), None
    for w1 in np.arange(0.0, 1.0 + step, step):
        w1  = round(w1, 4)
        w2  = round(1.0 - w1, 4)
        auc = roc_auc_score(y, w1 * oof_list[0] + w2 * oof_list[1])
        if auc > best_auc:
            best_auc  = auc
            best_w    = (w1, w2)
            best_test = w1 * test_list[0] + w2 * test_list[1]
    print(f'[{labels[0]}+{labels[1]}] OOF AUC: {best_auc:.5f}  '
          f'가중치: {labels[0]}={best_w[0]:.2f}, {labels[1]}={best_w[1]:.2f}')
    return best_test, best_w, best_auc

print('find_best_weights 정의 완료')


In [ ]:
print('Multi-Seed 앙상블 시작')
print(f'Seeds: {SEEDS}')
print(f'예상 소요: 약 50분 ~ 1시간\n')

lgbm_oof_ms, lgbm_test_ms, cat_oof_ms, cat_test_ms = run_multiseed_ensemble(
    SEEDS, LGBM_PARAMS_BASE, CATBOOST_PARAMS_BASE)

# LGBM + CatBoost 앙상블 (가중치 탐색)
ensemble_test, best_w, ensemble_auc = find_best_weights(
    oof_list  = [lgbm_oof_ms, cat_oof_ms],
    test_list = [lgbm_test_ms, cat_test_ms],
    y         = y,
    labels    = ['LGBM_multiseed', 'CatBoost_multiseed'],
)

lgbm_auc_ms = roc_auc_score(y, lgbm_oof_ms)
cat_auc_ms  = roc_auc_score(y, cat_oof_ms)


In [ ]:
print('\n' + '=' * 65)
print('             Day 4-8 최종 결과 요약')
print('=' * 65)
print(f'  [기준] v4 LB:               0.74191')
print(f'  [기준] v4 앙상블 OOF:       0.74040')
print(f'  Seeds: {SEEDS}')
print()
print(f'  LGBM  (5-seed 평균) OOF:    {lgbm_auc_ms:.5f}')
print(f'  CatBoost (5-seed 평균) OOF: {cat_auc_ms:.5f}')
print(f'  앙상블 OOF:                 {ensemble_auc:.5f}')
print(f'  가중치: LGBM={best_w[0]:.2f}, CatBoost={best_w[1]:.2f}')
print()
print(f'  v4 대비 앙상블 변화:        {ensemble_auc - 0.74040:+.5f}')
print('=' * 65)

if ensemble_auc > 0.74040:
    print('✅ v4 OOF 초과 → LB 0.74191 돌파 기대! 즉시 제출')
elif ensemble_auc > 0.74018:
    print('⚠️  day4_4 수준 → 제출해서 LB 확인')
else:
    print('❌ 하락 → Multi-seed 효과 없음')

# 저장
sample_sub = pd.read_csv(INPUT_DIR + 'sample_submission.csv')
fname = f'submission_day4_8_multiseed_OOF{ensemble_auc:.5f}.csv'
sample_sub['probability'] = ensemble_test
sample_sub.to_csv(OUTPUT_DIR + fname, index=False)
print(f'\n저장 완료: {fname}')

# seed별 결과를 따로 저장 (분석용)
np.save(OUTPUT_DIR + 'oof_lgbm_multiseed.npy',  lgbm_oof_ms)
np.save(OUTPUT_DIR + 'oof_cat_multiseed.npy',   cat_oof_ms)
np.save(OUTPUT_DIR + 'test_lgbm_multiseed.npy', lgbm_test_ms)
np.save(OUTPUT_DIR + 'test_cat_multiseed.npy',  cat_test_ms)
print('OOF/test 캐시 저장 완료')
